# 4b. Forced response figures: the quantiles and the width

The figures for the forced response, drawn from what notebook 03 saved, so nothing slow runs here. Notebook 03 explains each method, step by step. The figures are in four notebooks, so that none grows too big to open; each opens only the saved results it uses, so they can be run in any order after notebook 03:

| Notebook | Sections |
|---|---|
| 04a | 1. Mean change; 2. Signal to noise (and a check of every model's headline numbers) |
| 04b | 3. Quantile response; 4. Change in width, and which tail |
| 04c | 5. The whole distribution; 6. Additivity; 7. Zonal means and the sea-ice edge |
| 04d | 8. Probability ratios; 9. The shift-only ratio |

The sections keep one numbering across the four.

Every section shows its result in the same four figures, with the four seasons across every map:

1. **What goes in.** The quantities the result is made from, then the result, one row each: one model and experiment (`MODEL`, `EXPERIMENT`).
2. **One model.** The parts and the result for every experiment of `MODEL`.
3. **Every model.** The result for every model (rows), one figure per experiment, dotted where it is not significant.
4. **How many models.** The multi-model median of the result (top row), then at each grid point how many models have a significant increase and how many a significant decrease; one figure per experiment.

This notebook:

| Section | Result | Test |
|---|---|---|
| 3. Quantile response | the change in Q05, Q50 and Q95 | none |
| 4. Change in width | the change in Q95 − Q05, and which tail did it | hist-nat bootstrap |

**By SAM phase.** Section 4 ends with the result in the seasons of each phase of the Southern Annular Mode alone (negative, neutral and positive, cut at hist-nat's terciles of the index: notebook 03, *The SAM phases*), the phases as rows.

**Reading the figures.** A quantity has the same colour scale in every figure (set once, under *Colour scales*). Every change is red for an increase and blue for a decrease, and dots mark where a change is not significant. The black lines on the maps of section 4 are the sea-ice edge (the 15% contour of the ensemble-mean concentration over the final 21 years, from notebook 09): hist-nat's dashed, the experiment's solid; the multi-model figures draw the edge of the models' median concentration. Without the sea ice they are left out. In the counts, the colour changes where more than half the models agree: reds turn brown (an increase) and blues purple (a decrease). The paper figures, which put the models together, are in notebook 06. The cells before each figure pick out its data and show them.

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame). Each saved result has the same name as in notebook 03. Until 14 October 2026 the four were one notebook, `04_forced_response_figures.ipynb` (in the git history); the version before that, from 7 October, is in `archive/forced_response_figures_2026-10-07.ipynb`.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging

import pandas as pd
import xarray as xr

from extant import config, paths, response_change as rc, sam
from extant import significance as sig, storage
from extant.logger import setup_logging
from extant.plots import forced_response as fr, sea_ice as sip

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)
pd.set_option('display.max_rows', 200)   # every model, experiment and season in one table

In [ ]:
%matplotlib inline

## Settings

In [ ]:
VARIABLE = 'tas'
var = config.VARIABLES[VARIABLE]
UNITS = var.units

WINDOW = config.WINDOW

#(t): The experiments, in the order of every figure, and the seasons, the columns of every map
EXPERIMENTS = ['hist-GHG', 'hist-aer', 'hist-totalO3', 'historical']
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']

#(t): The model and experiment of the "what goes in" and "one model" figures
MODEL = config.PLOT_SEL['model']
EXPERIMENT = config.PLOT_SEL['experiment']

#(t): Significance level of every test
ALPHA = 0.05

#(t): The quantiles of section 3, and how they are labelled
RESPONSE_QUANTILES = [0.05, 0.50, 0.95]
QUANTILE_LABELS = {0.05: 'Q05', 0.50: 'Q50', 0.95: 'Q95'}

#(t): The SAM phases of 4.6, in the order of their rows, and how the rows are labelled
PHASES = list(sam.PHASES)
PHASE_LABELS = {phase: f'SAM\n{phase}' for phase in PHASES}

#(t): The share of the models that must have a significant change of one sign for a multi-model change to be robust
ROBUST = config.SIGNIFICANT_THRESHOLD


def result_path(name):
    """Where notebook 03 saved a result: results/<variable>/<run>/<name>.zarr."""
    return paths.result(name, VARIABLE, RUN)

### Colour scales

One scale per quantity, used by every figure that shows it (here and in notebooks 05 and 06), so the same colour always means the same value. Each is a `fr.Scale`: the contour levels, the colormap and the colour-bar label. Their ranges, per variable, are in `fr.RANGES`; notebook 06 (*Check the colour ranges*) says how much of each field falls beyond them.

In [ ]:
scales = fr.colour_scales(VARIABLE, alpha=ALPHA)

In [ ]:
#(t): What each one covers
pd.DataFrame({name: {'from': scale.levels[0], 'to': scale.levels[-1], 'colormap': getattr(scale.cmap, 'name', scale.cmap),
                     'label': scale.label}
              for name, scale in vars(scales).items()}).T

## Open the saved results

Run **one** of the next two sections. Both leave `RUN`, which says which saved results to open.

### Sample data

In [ ]:
#(t): The results notebook 03 saved from the sample
RUN = 'sample'

### Full data (JASMIN)

In [ ]:
#(t): The results notebook 03 saved from the full data
RUN = 'full'

### The saved results

Everything sections 3 and 4 use, from `results/<variable>/<RUN>/` under `paths.DATA_DIR` or `paths.SCRATCH` (the log gives each full path), loaded into memory. Grouped by the section that uses them.

In [ ]:
#(t): Section 3: hist-nat's and every experiment's final-years quantiles, and the change
hist_nat_final_q_da = storage.open_dataarray(result_path('hist_nat_final_quantiles'), load=True)
final_q_da = storage.open_dataarray(result_path('final_quantiles'), load=True)
quantile_response_da = storage.open_dataarray(result_path('quantile_response'), load=True)
quantile_curve_da = storage.open_dataarray(result_path('quantile_curve'), load=True)

#(t): Section 4: the quantiles of the internal variability, the width test, and the tails
width_quantiles_ds = storage.open_dataset(result_path('width_quantiles'), load=True)
qrange_change_ds = storage.open_dataset(result_path('qrange_change'), load=True)
tails_ds = storage.open_dataset(result_path('tails'), load=True)

#(t): By SAM phase (4.6): the width test and the tails in each phase
sam_qrange_change_ds = storage.open_dataset(result_path('sam_qrange_change'), load=True)
sam_tails_ds = storage.open_dataset(result_path('sam_tails'), load=True)

In [ ]:
#(t): What each result holds: its dims and its variables
results = {
    'hist_nat_final_q_da': hist_nat_final_q_da, 'final_q_da': final_q_da,
    'quantile_response_da': quantile_response_da, 'quantile_curve_da': quantile_curve_da,
    'width_quantiles_ds': width_quantiles_ds, 'qrange_change_ds': qrange_change_ds, 'tails_ds': tails_ds,
    'sam_qrange_change_ds': sam_qrange_change_ds, 'sam_tails_ds': sam_tails_ds,
}
pd.DataFrame({
    name: {'dims': dict(result.sizes), 'variables': ', '.join(result.data_vars) if isinstance(result, xr.Dataset) else ''}
    for name, result in results.items()
}).T

In [ ]:
#(t): The number of models: the top of the colour bar of every count figure
N_MODELS = qrange_change_ds.sizes['model']
N_MODELS

### Sea ice

The sea-ice edge on the maps of section 4 comes from notebook 09, section 3. Run **one** of the next two sections. Both leave `ice_climatology_da` (the ensemble-mean concentration over the final years), and `model_ice_da` and `median_ice_da` (`MODEL`'s concentration, and the models' median, for the edges of the one-model and multi-model maps); without sea ice they are `None` and the figures leave the edge out.

#### Without sea ice

Before notebook 09 has saved the sea ice (on the sample, that is when the notebooks are run in order).

In [ ]:
ice_climatology_da = model_ice_da = median_ice_da = None

#### With sea ice (JASMIN)

In [ ]:
#(t): What notebook 09 saved, for the same RUN (on the sample too, once notebook 09 has run on it)
ice_climatology_da = storage.open_dataarray(paths.result('climatology', 'siconc', RUN), load=True)
model_ice_da = ice_climatology_da.sel(model=MODEL)
median_ice_da = ice_climatology_da.median('model')

## 3. Quantile response: Q05, Q50 and Q95

How each part of the distribution has moved: the quantiles of each experiment's final 21 years (members and years pooled) minus hist-nat's over the same years (notebook 03, section 3):

$$
\Delta Q_p = Q_p\left(x^{\mathrm{exp}}_{\text{final } L \text{ years}}\right) - Q_p\left(x^{\mathrm{nat}}_{\text{final } L \text{ years}}\right).
$$

These are the values themselves, so the shift of the whole distribution is in every quantile. Equal changes in Q05, Q50 and Q95 are a pure shift; a bigger change in Q95 than in Q05 means the distribution has widened. There is no test of a single quantile: section 4 tests the width, Q95 − Q05.

### 3.1 What goes in

hist-nat's Q05, Q50 and Q95 and `EXPERIMENT`'s, both over the final 21 years, `MODEL`, and the change in each.

In [ ]:
#(t): hist-nat's quantiles over the final WINDOW years, MODEL, every season
hist_nat_q_da = hist_nat_final_q_da.sel(model=MODEL, season=SEASONS, quantile=RESPONSE_QUANTILES)
hist_nat_q_da

In [ ]:
#(t): EXPERIMENT's over the same years
experiment_q_da = final_q_da.sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS, quantile=RESPONSE_QUANTILES)
experiment_q_da

In [ ]:
#(t): The change in each (quantile_response_da holds the same)
quantile_change_da = experiment_q_da - hist_nat_q_da
quantile_change_da

In [ ]:
panels = fr.map_grid(hist_nat_q_da, scales.temperature,
                     title=f'What goes into the quantile response: hist-nat (final {WINDOW} years), {MODEL}',
                     row_dim='quantile', col_dim='season', row_labels=QUANTILE_LABELS)

In [ ]:
panels = fr.map_grid(experiment_q_da, scales.temperature,
                     title=f'What goes into the quantile response: {EXPERIMENT} (final {WINDOW} years), {MODEL}',
                     row_dim='quantile', col_dim='season', row_labels=QUANTILE_LABELS)

In [ ]:
panels = fr.map_grid(quantile_change_da, scales.change,
                     title=f'The quantile response: {MODEL}, {EXPERIMENT} (final {WINDOW} years) against '
                           f'hist-nat (the same years)',
                     row_dim='quantile', col_dim='season', row_labels=QUANTILE_LABELS)

### 3.2 One model

The change in Q05, Q50 and Q95 for every experiment of `MODEL`.

In [ ]:
#(t): MODEL's change in each quantile, every experiment and season
model_quantile_change_da = quantile_response_da.sel(model=MODEL, experiment=EXPERIMENTS, season=SEASONS)
model_quantile_change_da

In [ ]:
for quantile in RESPONSE_QUANTILES:
    panels = fr.map_grid(
        model_quantile_change_da.sel(quantile=quantile), scales.change,
        title=f'{MODEL}: change in {QUANTILE_LABELS[quantile]}, final {WINDOW} years against hist-nat',
        row_dim='experiment', col_dim='season',
    )

### 3.3 Every model

The two ends of the distribution, Q05 and Q95 (the median moves with the mean, section 1); one figure per experiment.

In [ ]:
#(t): The change in Q05 for every model, experiment and season
all_q05_change_da = quantile_response_da.sel(quantile=0.05, experiment=EXPERIMENTS, season=SEASONS)
all_q05_change_da

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.map_grid(
        all_q05_change_da.sel(experiment=experiment), scales.change, col_dim='season',
        title=f'Change in Q05, final {WINDOW} years against hist-nat, every model: {experiment}',
    )

In [ ]:
#(t): and in Q95
all_q95_change_da = quantile_response_da.sel(quantile=0.95, experiment=EXPERIMENTS, season=SEASONS)
all_q95_change_da

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.map_grid(
        all_q95_change_da.sel(experiment=experiment), scales.change, col_dim='season',
        title=f'Change in Q95, final {WINDOW} years against hist-nat, every model: {experiment}',
    )

### 3.4 Every model, every quantile

With no test there is nothing to count, so instead the change in every 5th percentile, averaged over the area (south of 40°S), one line per model; experiments down, seasons across. A flat line is a pure shift; a rising line means the upper quantiles moved more than the lower ones, so the distribution widened.

In [ ]:
#(t): The area-mean change in every 5th percentile, every model, experiment and season
all_quantile_curve_da = quantile_curve_da.sel(experiment=EXPERIMENTS, season=SEASONS)
all_quantile_curve_da

In [ ]:
panels = fr.quantile_curves(
    all_quantile_curve_da,
    title=f'Change at every 5th percentile, area mean south of {abs(config.LAT_MAX)}°S, every model',
    units=UNITS,
    row_dim='experiment', col_dim='season',
)

## 4. Change in width, and which tail

The width is $W = Q_{95} - Q_{05}$ of the **internal variability**: each member minus its experiment's forced response (the ensemble median smoothed with LOWESS), so that a trend within the 21 years does not widen it. The change is the width of each experiment's final 21 years minus hist-nat's over the same years (notebook 03, section 2):

$$
\Delta W = W\left(x'^{\,\mathrm{exp}}_{\text{final } L \text{ years}}\right) - W\left(x'^{\,\mathrm{nat}}_{\text{final } L \text{ years}}\right).
$$

It is tested against 1000 hist-nat bootstrap trials, each the width of 10 random hist-nat members over the same 21 years: significant where the experiment's width is outside the trials' central 95% ($p < 0.05$).

**Which tail.** The median splits the width into the lower tail $Q_{50} - Q_{05}$ and the upper tail $Q_{95} - Q_{50}$. Their changes add up to $\Delta W$, so the larger one says which tail did the widening or narrowing; for temperature, the lower tail is the cold one. Only the width is tested.

### 4.1 What goes in

The two tails and the width of hist-nat and of `EXPERIMENT`, both over the final 21 years, and the change in each; `MODEL`.

In [ ]:
#(t): hist-nat's Q05, Q50 and Q95 of the internal variability, over the final WINDOW years, MODEL, every season
hist_nat_width_q_da = width_quantiles_ds['hist_nat'].sel(model=MODEL, season=SEASONS)
hist_nat_width_q_da

In [ ]:
#(t): EXPERIMENT's, over the same years
experiment_width_q_da = width_quantiles_ds['final'].sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS)
experiment_width_q_da

In [ ]:
#(t): hist-nat's lower tail Q50 - Q05, upper tail Q95 - Q50 and width Q95 - Q05, on a new 'part' dim
hist_nat_parts_da = rc.width_parts(hist_nat_width_q_da)
hist_nat_parts_da

In [ ]:
#(t): EXPERIMENT's
experiment_parts_da = rc.width_parts(experiment_width_q_da)
experiment_parts_da

In [ ]:
#(t): The change in each (tails_ds and qrange_change_ds hold the same)
parts_change_da = experiment_parts_da - hist_nat_parts_da
parts_change_da

In [ ]:
#(t): How the rows are labelled
PART_LABELS = {part: f'{part.capitalize()}\n{formula}' for part, formula in rc.WIDTH_PARTS.items()}
PART_LABELS

In [ ]:
panels = fr.map_grid(hist_nat_parts_da, scales.spread,
                     title=f'What goes into the change in width: hist-nat (final {WINDOW} years), {MODEL}, '
                           f'internal variability',
                     row_dim='part', col_dim='season', row_labels=PART_LABELS)

In [ ]:
panels = fr.map_grid(experiment_parts_da, scales.spread,
                     title=f'What goes into the change in width: {EXPERIMENT} (final {WINDOW} years), {MODEL}, '
                           f'internal variability',
                     row_dim='part', col_dim='season', row_labels=PART_LABELS)

In [ ]:
panels = fr.map_grid(parts_change_da, scales.spread_change,
                     title=f'The change in width: {MODEL}, {EXPERIMENT} (final {WINDOW} years) against hist-nat '
                           f'(the same years), internal variability',
                     row_dim='part', col_dim='season', row_labels=PART_LABELS)

The test of that change: it is significant where it falls outside the central 95% of the hist-nat trials (the 2.5th to the 97.5th percentile of their changes).

In [ ]:
#(t): The width test for MODEL and EXPERIMENT: the change, the trials' range, the p-value
width_test_ds = qrange_change_ds.sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS)
width_test_ds

In [ ]:
#(t): Not significant, where there is data
width_not_significant_da = ~width_test_ds['qrange_significant'] & width_test_ds['qrange_change'].notnull()

In [ ]:
panels = fr.row_grid(
    [fr.Row(width_test_ds['qrange_change'], scales.spread_change, 'Change in width',
            not_significant=width_not_significant_da, edges=sip.edge_pair(model_ice_da, EXPERIMENT)),
     fr.Row(width_test_ds['null_lower'], scales.spread_change, 'hist-nat trials\n2.5th percentile'),
     fr.Row(width_test_ds['null_upper'], scales.spread_change, 'hist-nat trials\n97.5th percentile'),
     fr.Row(width_test_ds['qrange_pvalue'], scales.pvalue, 'p-value\nhist-nat bootstrap')],
    title=f'The width test: {MODEL}, {EXPERIMENT}',
    not_significant_label=f'Not significant (inside the trials\' range, p ≥ {ALPHA:g})',
)

### 4.2 One model

For every experiment of `MODEL`: the change in each tail and the change in the width (their sum), dotted where the width test is not significant (4.1 shows the p-values).

In [ ]:
#(t): MODEL's tails, every experiment and season
model_tails_ds = tails_ds.sel(model=MODEL, experiment=EXPERIMENTS, season=SEASONS)
model_tails_ds

In [ ]:
#(t): Its width test
model_width_test_ds = qrange_change_ds.sel(model=MODEL, experiment=EXPERIMENTS, season=SEASONS)
model_width_test_ds

In [ ]:
#(t): Not significant, where there is data
model_width_not_significant_da = ~model_width_test_ds['qrange_significant'] & model_width_test_ds['qrange_change'].notnull()

In [ ]:
panels = fr.map_grid(model_tails_ds['lower_tail_change'], scales.spread_change,
                     title=f'{MODEL}: change in the lower tail Q50 − Q05, final {WINDOW} years against hist-nat',
                     row_dim='experiment', col_dim='season', edges=sip.edge_pair(model_ice_da, EXPERIMENTS))

In [ ]:
panels = fr.map_grid(model_tails_ds['upper_tail_change'], scales.spread_change,
                     title=f'{MODEL}: change in the upper tail Q95 − Q50, final {WINDOW} years against hist-nat',
                     row_dim='experiment', col_dim='season', edges=sip.edge_pair(model_ice_da, EXPERIMENTS))

In [ ]:
panels = fr.map_grid(model_width_test_ds['qrange_change'], scales.spread_change,
                     title=f'{MODEL}: change in the width Q95 − Q05, final {WINDOW} years against hist-nat',
                     row_dim='experiment', col_dim='season',
                     not_significant=model_width_not_significant_da,
                     not_significant_label=f'Not significant (p ≥ {ALPHA:g})',
                     edges=sip.edge_pair(model_ice_da, EXPERIMENTS))

### 4.3 Every model

One figure per experiment: every model down, every season across.

In [ ]:
#(t): The change in width for every model, experiment and season
all_width_change_da = qrange_change_ds['qrange_change'].sel(experiment=EXPERIMENTS, season=SEASONS)
all_width_change_da

In [ ]:
#(t): Significant (p < ALPHA in notebook 03)
all_width_significant_da = qrange_change_ds['qrange_significant'].sel(experiment=EXPERIMENTS, season=SEASONS)

In [ ]:
#(t): Not significant, where there is data
all_width_not_significant_da = ~all_width_significant_da & all_width_change_da.notnull()

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.map_grid(
        all_width_change_da.sel(experiment=experiment), scales.spread_change, col_dim='season',
        title=f'Change in width Q95 − Q05, final {WINDOW} years against hist-nat, every model: {experiment}',
        not_significant=all_width_not_significant_da.sel(experiment=experiment),
        not_significant_label=f'Not significant (hist-nat bootstrap, p ≥ {ALPHA:g})',
        edges=sip.edge_pair(ice_climatology_da, experiment),
    )

### 4.4 How many models

The multi-model median of the change in width (top row), then the number of models with a significantly wider distribution and a significantly narrower one. One figure per experiment.

In [ ]:
#(t): The multi-model median of the change in width
median_width_change_da = all_width_change_da.median('model')
median_width_change_da

In [ ]:
#(t): How many models are significantly wider, and how many narrower
width_counts_da = sig.significant_counts(all_width_change_da, all_width_significant_da)
width_counts_da

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.count_grid(
        width_counts_da.sel(experiment=experiment), N_MODELS,
        above=[fr.Row(median_width_change_da.sel(experiment=experiment), scales.spread_change, 'Multi-model\nmedian',
                      edges=sip.edge_pair(median_ice_da, experiment))],
        title=f'How many models have a significant change in width: {experiment}',
        label=f'Number of models significantly narrower or wider, p < {ALPHA:g}',
        col_dim='season',
        row_labels={'increase': 'Significantly\nwider', 'decrease': 'Significantly\nnarrower'},
        edges=sip.edge_pair(median_ice_da, experiment),
    )

### 4.5 Which tail, every model

The upper tail's change minus the lower tail's: red where the upper (warm) tail stretched more or shrank less, blue where the lower (cold) tail did. Dotted where the width change itself is not significant; one figure per experiment.

In [ ]:
#(t): The tail asymmetry for every model, experiment and season
all_tail_asymmetry_da = tails_ds['tail_asymmetry'].sel(experiment=EXPERIMENTS, season=SEASONS)
all_tail_asymmetry_da

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.map_grid(
        all_tail_asymmetry_da.sel(experiment=experiment), scales.tail, col_dim='season',
        title=f'Which tail changed more: upper-tail minus lower-tail change, every model: {experiment}',
        not_significant=all_width_not_significant_da.sel(experiment=experiment),
        not_significant_label=f'Width change not significant (p ≥ {ALPHA:g})',
        edges=sip.edge_pair(ice_climatology_da, experiment),
    )

### 4.6 By SAM phase

The change in width in the seasons of each SAM phase alone (notebook 03, section 2, *By SAM phase*), and which tail did it (notebook 03, section 7): for `MODEL`, dotted where it is not significant; the multi-model median, dotted where it is not robust; how many models; and the multi-model median of which tail.

In [ ]:
#(t): MODEL's change in width in each phase, every experiment and season
model_phase_width_change_da = sam_qrange_change_ds['qrange_change'].sel(model=MODEL, experiment=EXPERIMENTS,
                                                                         season=SEASONS)
model_phase_width_change_da

In [ ]:
#(t): Not significant, where there is data
model_phase_width_not_significant_da = (~sam_qrange_change_ds['qrange_significant'].sel(model=MODEL, experiment=EXPERIMENTS,
                                                                                         season=SEASONS)
                                        & model_phase_width_change_da.notnull())

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.map_grid(
        model_phase_width_change_da.sel(experiment=experiment), scales.spread_change,
        title=f'{MODEL}: change in width Q95 − Q05 in each SAM phase, final {WINDOW} years against hist-nat: {experiment}',
        row_dim='phase', col_dim='season', row_labels=PHASE_LABELS,
        not_significant=model_phase_width_not_significant_da.sel(experiment=experiment),
        not_significant_label=f'Not significant (p ≥ {ALPHA:g})',
    )

In [ ]:
#(t): The change in width in each phase, every model, experiment and season
all_phase_width_change_da = sam_qrange_change_ds['qrange_change'].sel(experiment=EXPERIMENTS, season=SEASONS)
all_phase_width_change_da

In [ ]:
#(t): Significant (p < ALPHA in notebook 03)
all_phase_width_significant_da = sam_qrange_change_ds['qrange_significant'].sel(experiment=EXPERIMENTS, season=SEASONS)

In [ ]:
#(t): The multi-model median
median_phase_width_change_da = all_phase_width_change_da.median('model')
median_phase_width_change_da

In [ ]:
#(t): Where the models agree: +1 (-1) where at least ROBUST of them are significantly wider (narrower)
phase_width_robust_da = sig.robust_sign(all_phase_width_change_da, all_phase_width_significant_da, threshold=ROBUST)
phase_width_robust_da

In [ ]:
#(t): Not robust, where there is data
phase_width_not_robust_da = phase_width_robust_da == 0

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.map_grid(
        median_phase_width_change_da.sel(experiment=experiment), scales.spread_change,
        title=f'Change in width Q95 − Q05 in each SAM phase, multi-model median: {experiment}',
        row_dim='phase', col_dim='season', row_labels=PHASE_LABELS,
        not_significant=phase_width_not_robust_da.sel(experiment=experiment),
        not_significant_label='Not robust (fewer than two-thirds of the models significant with one sign)',
    )

In [ ]:
#(t): How many models are significantly wider, and how many narrower, in each phase
phase_width_counts_da = sig.significant_counts(all_phase_width_change_da, all_phase_width_significant_da)
phase_width_counts_da

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.count_grid(
        phase_width_counts_da.sel(experiment=experiment), N_MODELS,
        title=f'How many models have a significant change in width in each SAM phase: {experiment}',
        label=f'Number of models significantly narrower or wider, p < {ALPHA:g}',
        col_dim='season', group_dim='phase',
        row_labels={'increase': 'Significantly\nwider', 'decrease': 'Significantly\nnarrower'},
    )

In [ ]:
#(t): Which tail: the multi-model median of the upper tail's change minus the lower tail's, in each phase
median_phase_tail_asymmetry_da = sam_tails_ds['tail_asymmetry'].sel(experiment=EXPERIMENTS, season=SEASONS).median('model')
median_phase_tail_asymmetry_da

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.map_grid(
        median_phase_tail_asymmetry_da.sel(experiment=experiment), scales.tail,
        title=f'Which tail changed more in each SAM phase: upper-tail minus lower-tail change, multi-model median: '
              f'{experiment}',
        row_dim='phase', col_dim='season', row_labels=PHASE_LABELS,
        not_significant=phase_width_not_robust_da.sel(experiment=experiment),
        not_significant_label='Width change not robust',
    )